In [30]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
import shap
import joblib
from sklearn.model_selection import train_test_split

In [31]:
best_model = joblib.load(r'D:\IT\Python\sklearn\hotel-booking-project\models\xgb_model.joblib')
df = pd.read_csv(r'D:\IT\Python\sklearn\hotel-booking-project\data\processed_hotel_booking.csv')
X = df.drop("is_canceled", axis=1)
y = df['is_canceled']
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

In [36]:
model = best_model.named_steps["model"]
prep = best_model.named_steps["prep"]

feature_names = prep.get_feature_names_out()
feature_names = [col.replace("num__", "").replace("cat__", "") for col in feature_names]
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

importance_df.head(20)

,feature,importance
247,deposit_type_Non Refund,0.497859
259,room_changed_0,0.059331
14,required_car_parking_spaces,0.042663
217,market_segment_Online TA,0.042575
168,country_PRT,0.018919
9,previous_cancellations,0.016042
246,deposit_type_No Deposit,0.012184
15,total_of_special_requests,0.010424
251,customer_type_Transient,0.009871
214,market_segment_Direct,0.008711


In [32]:
i = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring='roc_auc'
)

In [35]:
pd.Series(
    i.importances_mean,
    index=X_test.columns
).sort_values(ascending=False)

country                           0.085095
deposit_type                      0.073919
market_segment                    0.064041
lead_time                         0.043021
required_car_parking_spaces       0.037350
total_of_special_requests         0.030695
previous_cancellations            0.025948
arrival_date_year                 0.021037
room_changed                      0.020054
adr                               0.019882
customer_type                     0.019754
arrival_date_week_number          0.006414
booking_changes                   0.005317
hotel                             0.004597
previous_bookings_not_canceled    0.004432
arrival_date_month                0.002336
meal                              0.001962
total_nights                      0.001562
distribution_channel              0.001380
arrival_date_day_of_month         0.001346
assigned_room_type                0.001235
has_agent                         0.000848
days_in_waiting_list              0.000832
stays_in_we

In [56]:
import shap

explainer = shap.TreeExplainer(model)

X_transformed = prep.transform(X_test).toarray()

print(type(X_transformed))
print(X_transformed.shape)

if hasattr(X_transformed, "toarray"):
    X_transformed = X_transformed.toarray()

X_transformed = pd.DataFrame(
    X_transformed,
    columns=feature_names
)

print(X_transformed.shape)

shap_values = explainer(X_transformed)

<class 'numpy.ndarray'>
(23878, 212)


ValueError: Shape of passed values is (23878, 212), indices imply (23878, 261)

In [54]:
print(X_transformed.shape)
print(len(feature_names))

(23878, 212)
261
